In [1]:
# Cell 1: Environment Setup

include("scripts/buildaux_helpers.jl")
include("scripts/buildaux_dictionaries.jl")
using .BuildAuxHelpers
using .BuildAuxDictionaries
using OMJulia

# --- Configuration ---

# 1. Directory containing the single-file model
MODEL_DIR = abspath("models")

# 2. Select the model to build
MODEL = "MyBESS"

# 3. Path to the selected model file
MODEL_FILE_PATH = joinpath(MODEL_DIR, MODEL * ".mo")

# 4. Path to the Dynawo package.mo
DYNAWO_PKG_PATH   = "/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"

# 5. Path to the Modelica package.mo
MODELICA_PKG_PATH = "/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"


"/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"

In [2]:
# Cell 2: OpenModelica Setup + Single Model Validation

# 1. Start OMC and load libraries
omc = OMJulia.OMCSession()
om_send(omc, "loadFile(\"$MODELICA_PKG_PATH\")")
om_send(omc, "loadModel(Complex)")
om_send(omc, "loadModel(ModelicaServices)")
om_send(omc, "loadFile(\"$DYNAWO_PKG_PATH\")")

# 2. Load the selected model and validate it
om_send(omc, "loadFile(\"$MODEL_FILE_PATH\")")
om_send(omc, "clearMessages()")
chk = om_send(omc, "checkModel($MODEL)", parsed=false)
println(chk)


[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.R4c0zVb5MW"


OMC -> loadFile("/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo")
OMC -> loadModel(Complex)
OMC -> loadModel(ModelicaServices)
OMC -> loadFile("/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo")
OMC -> loadFile("/home/clarafercas/dynawo-notebooks/OpenModelica_only_users/BuildAux/models/MyBESS.mo")
OMC -> clearMessages()
OMC -> checkModel(MyBESS)
"Check of MyBESS completed successfully.
Class MyBESS has 559 equation(s) and 559 variable(s).
389 of these are trivial equation(s)."



In [3]:
# Cell 3: Auxiliary Model Setup

AUX_MODEL = MODEL * "_auxiliary"
AUX_FILE = joinpath(MODEL_DIR, AUX_MODEL * ".mo")


"/home/clarafercas/dynawo-notebooks/OpenModelica_only_users/BuildAux/models/MyBESS_auxiliary.mo"

In [4]:
# Cell 4: INIT / Optional Slack Configuration for the Single Model

INIT_MODEL_BY_COMPONENT = Dict{String, String}(
 #"generatorSynchronous" => "GeneratorSynchronousInt_INIT",
)

# Leave empty to disable slack-specific handling.
SLACK_COMPONENT = "inertialGrid1"


"inertialGrid1"

In [ ]:
# Cell 5: Single-Model Auxiliary Build Pipeline

# Create/refresh the auxiliary model in OpenModelica
om_send(omc, "deleteClass($AUX_MODEL)")
om_send(omc, "clearMessages()")
om_send(omc, "copyClass($MODEL, \"$AUX_MODEL\")")

# Build component dictionary from the source model
components = get_all_components(omc, MODEL)

# Apply dictionary-driven replacements
apply_replacements!(omc, MODEL, AUX_MODEL, REPLACEMENTS, components, SLACK_COMPONENT)

# Delete connections to cleanup targets
delete_connections!(omc, AUX_MODEL, components)

# Delete cleanup-target components
delete_components!(omc, AUX_MODEL, components)

# Add INIT models for the source model
add_init_models!(omc, MODEL, AUX_MODEL, INIT_MODELS, INIT_MODEL_BY_COMPONENT, components, SLACK_COMPONENT)

# Add load-flow modifiers
apply_LF_modifiers!(omc, MODEL, AUX_MODEL, INIT_MODELS, components)

# Add initial equations for the source model
add_init_equations!(omc, MODEL, AUX_MODEL, components, INIT_MODELS, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)

# Clean up the auxiliary model equations in place
clean_aux_equations!(omc, AUX_MODEL, SLACK_COMPONENT; components = components)

# Validate the build
om_send(omc, "clearMessages()")
chk = om_send(omc, "checkModel($AUX_MODEL)", parsed=false)
println(chk)

# Save the auxiliary model
om_send(omc, "saveModel(\"$AUX_FILE\", $AUX_MODEL)")


### Optional diagnostics for the single-file build
Run the next cell only if the main build/check cell fails or you need detailed OpenModelica messages.


In [6]:
# Cell 7: OMC diagnostics for failed checks

# Run this cell after the build/check cell to isolate OpenModelica failures.

function _print_omc_errors(label::String)
    raw = String(sendExpression(omc, "getErrorString()", parsed=false))
    txt = strip(replace(raw, "\"" => ""))
    println("\n[$label] getErrorString()")
    if isempty(txt)
        println("<no messages>")
    else
        println(raw)
    end
end

function _check_and_report(model_name::String)
    sendExpression(omc, "clearMessages()")
    println("\n=== checkModel($model_name) ===")
    chk = sendExpression(omc, "checkModel($model_name)", parsed=false)
    println(chk)
    _print_omc_errors(model_name)
    return chk
end

println("=== OMC diagnostics start ===")
_print_omc_errors("after previous cell")

# 1) Auxiliary model
_check_and_report(AUX_MODEL)

# 2) Optional compile-time expansion (often gives clearer errors)
sendExpression(omc, "clearMessages()")
println("\n=== instantiateModel($AUX_MODEL) ===")
inst = sendExpression(omc, "instantiateModel($AUX_MODEL)", parsed=false)
inst_s = String(inst)
if startswith(strip(inst_s), "Error")
    println(inst_s)
end
_print_omc_errors("instantiateModel")

println("=== OMC diagnostics end ===")


=== OMC diagnostics start ===

[after previous cell] getErrorString()
"[/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/Electrical/Controls/Basics/SwitchOff/SwitchOffLogic.mo:20:3-20:98:writable] Warning: Connector switchOffSignal1 is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/Electrical/Controls/Basics/SwitchOff/SwitchOffLogic.mo:21:3-21:125:writable] Warning: Connector switchOffSignal2 is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/Electrical/Controls/Basics/SwitchOff/SwitchOffLogic.mo:22:3-22:125:writable] Warning: Connector switchOffSignal3 is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[/home/clarafercas/Model_library/dynawo/dy

In [7]:
# Cell 8: Trace load INIT mode selection

# Run this cell to inspect how BuildAux classifies each Dynawo load in MODEL.

components = get_all_components(omc, MODEL)
found_load = false

println("=== Load INIT mode trace for $MODEL ===")

for (comp_name, c) in sort(collect(components), by = first)
    base_class = c["class"]::String
    startswith(base_class, "Dynawo.Electrical.Loads.") || continue

    found_load = true
    mode = BuildAuxBuild._load_init_mode(omc, MODEL, components, comp_name, base_class)

    raw_mods = c["modifiers"]
    has_direct_complex =
        isa(raw_mods, Dict) &&
        haskey(raw_mods, "s0Pu") &&
        haskey(raw_mods, "u0Pu") &&
        haskey(raw_mods, "i0Pu")

    p_ref = strip(resolve_load_ref_value(omc, MODEL, comp_name, "PRefPu"))
    q_ref = strip(resolve_load_ref_value(omc, MODEL, comp_name, "QRefPu"))

    println("\n$comp_name")
    println("  class = $base_class")
    println("  mode = $mode")
    println("  has s0Pu/u0Pu/i0Pu = $has_direct_complex")
    println("  PRefPu source = ", isempty(p_ref) ? "<none>" : p_ref)
    println("  QRefPu source = ", isempty(q_ref) ? "<none>" : q_ref)
end

if !found_load
    println("No Dynawo load components found in $MODEL")
end

println("\n=== End load INIT mode trace ===")


=== Load INIT mode trace for MyBESS ===
No Dynawo load components found in MyBESS

=== End load INIT mode trace ===


In [8]:
# Cell 9: Standalone diagnostics for the original source model

# This cell is self-contained: you can run it first, without running any
# other notebook cell.

using OMJulia

function _pick_existing_file(label::String, candidates::Vector{String})
    for p in candidates
        isempty(strip(p)) && continue
        if isfile(p)
            println(label, " = ", p)
            return p
        end
    end
    println(label, " candidates checked:")
    for p in candidates
        isempty(strip(p)) && continue
        println("  ", p, "  [", isfile(p) ? "found" : "missing", "]")
    end
    error("Could not find a valid file for " * label)
end

function _pick_existing_dir(label::String, candidates::Vector{String})
    for p in candidates
        isempty(strip(p)) && continue
        if isdir(p)
            println(label, " = ", p)
            return p
        end
    end
    println(label, " candidates checked:")
    for p in candidates
        isempty(strip(p)) && continue
        println("  ", p, "  [", isdir(p) ? "found" : "missing", "]")
    end
    error("Could not find a valid directory for " * label)
end

MODEL = isdefined(Main, :MODEL) ? String(Main.MODEL) : "MyPVCurrent"

model_dir_candidates = unique([
    isdefined(Main, :MODEL_DIR) ? String(Main.MODEL_DIR) : "",
    abspath("models"),
    abspath(joinpath("OpenModelica_only_users", "BuildAux", "models")),
    abspath(joinpath("dynawo-notebooks", "OpenModelica_only_users", "BuildAux", "models")),
])

MODEL_DIR = _pick_existing_dir("MODEL_DIR", model_dir_candidates)
MODEL_FILE_PATH = joinpath(MODEL_DIR, MODEL * ".mo")
isfile(MODEL_FILE_PATH) || error("Model file not found: " * MODEL_FILE_PATH)
println("MODEL = ", MODEL)
println("MODEL_FILE_PATH = ", MODEL_FILE_PATH)

modelica_candidates = unique([
    isdefined(Main, :MODELICA_PKG_PATH) ? String(Main.MODELICA_PKG_PATH) : "",
    "/home/dyvulgawocfc/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo",
    "/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo",
])

dynawo_candidates = unique([
    isdefined(Main, :DYNAWO_PKG_PATH) ? String(Main.DYNAWO_PKG_PATH) : "",
    "/home/dyvulgawocfc/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo",
    "/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo",
])

MODELICA_PKG_PATH = _pick_existing_file("MODELICA_PKG_PATH", modelica_candidates)
DYNAWO_PKG_PATH = _pick_existing_file("DYNAWO_PKG_PATH", dynawo_candidates)

diag_omc = OMJulia.OMCSession()

function _show_messages(omc)
    raw = String(sendExpression(omc, "getErrorString()", parsed=false))
    txt = strip(replace(raw, "\"" => ""))
    println(isempty(txt) ? "<no messages>" : raw)
end

function _run_step(omc, expr; parsed=true)
    sendExpression(omc, "clearMessages()")
    println("\n=== ", expr, " ===")
    result = try
        sendExpression(omc, expr; parsed=parsed)
    catch err
        println("Julia/OMJulia exception:")
        showerror(stdout, err)
        println()
        nothing
    end
    println("result = ", repr(result))
    println("[OMC messages]")
    _show_messages(omc)
    return result
end

println("=== Standalone source-model diagnostics start ===")

_run_step(diag_omc, "loadFile(\"$MODELICA_PKG_PATH\")")
_run_step(diag_omc, "loadModel(Complex)")
_run_step(diag_omc, "loadModel(ModelicaServices)")
_run_step(diag_omc, "loadFile(\"$DYNAWO_PKG_PATH\")")
_run_step(diag_omc, "loadFile(\"$MODEL_FILE_PATH\")")
_run_step(diag_omc, "checkModel($MODEL)", parsed=false)
_run_step(diag_omc, "instantiateModel($MODEL)", parsed=false)
_run_step(diag_omc, "getComponentCount($MODEL)")

println("\n=== First lines of the model file ===")
for (i, line) in enumerate(split(read(MODEL_FILE_PATH, String), '\n')[1:min(end, 25)])
    println(lpad(i, 3), ": ", line)
end

println("=== Standalone source-model diagnostics end ===")


[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.nMVFLidDCd"


MODEL_DIR = /home/clarafercas/dynawo-notebooks/OpenModelica_only_users/BuildAux/models
MODEL = MyBESS
MODEL_FILE_PATH = /home/clarafercas/dynawo-notebooks/OpenModelica_only_users/BuildAux/models/MyBESS.mo
MODELICA_PKG_PATH = /home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo
DYNAWO_PKG_PATH = /home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo
=== Standalone source-model diagnostics start ===

=== loadFile("/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo") ===
result = true
[OMC messages]
"Notification: Automatically loaded package Complex 4.1.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.1.0 due to uses annotation from Modelica.
"


=== loadModel(Complex) ===
result = true
[OMC messages]
<no messages>

=== loadModel(ModelicaServices) ===
result = true
[OMC messages]
<no messages>

=== loadFile("/home/clarafercas/Model_library/dynawo/dynawo/sources/Mo